# Text-to-SVG V3: Qwen3.5-2B via Unsloth (Colab)

**NYU Deep Learning Spring 2026 — Kaggle Competition**

**Environment:** Google Colab Pro (A100 GPU) + Google Drive

### Setup Instructions
1. Upload `train.csv` and `test.csv` to your Google Drive under `MyDrive/svg-competition/`
2. Set Runtime → Change runtime type → **GPU** (A100 if available)
3. Run all cells in order
4. Adapter weights will be saved to `MyDrive/svg-competition/svg-lora-adapter/`

### V3 — Unsloth variant
- **Model: `unsloth/Qwen3.5-2B-Instruct-bnb-4bit`** (same as starter notebook)
- 2B params, newest Qwen generation, no thinking mode
- Unsloth provides optimized kernels → faster training
- **Consistent prompt** for training AND inference
- **Normalized training SVGs** to 256×256 canvas
- **Strict validation** — every output guaranteed valid
- **`<svg` prefill** during inference

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/svg-competition'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Contents: {os.listdir(PROJECT_DIR)}')

In [ ]:
!pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml cairosvg

In [ ]:
import os, re, time, random, json
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Configuration

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/svg-competition'

SYSTEM_PROMPT = 'Generate SVG code.'

ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter'
}

CONFIG = {
    'model_name': 'unsloth/Qwen3.5-2B',
    'max_seq_length': 2048,
    'lora_r': 32,
    'lora_alpha': 32,
    'learning_rate': 2e-4,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 16,
    'gradient_accumulation_steps': 1,     # Effective batch = 16
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'logging_steps': 20,
    'save_steps': 500,
    'eval_steps': 500,
    'save_total_limit': 3,
    'output_dir': '/content/svg-lora-checkpoints',
    'train_csv': f'{PROJECT_DIR}/train.csv',
    'test_csv': f'{PROJECT_DIR}/test.csv',
    'eval_fraction': 0.02,
    'max_svg_chars': 16000,
    'adapter_save_dir': f'{PROJECT_DIR}/svg-lora-adapter-v3-unsloth',
}

for key in ['train_csv', 'test_csv']:
    print(f'{key}: {"FOUND" if os.path.exists(CONFIG[key]) else "NOT FOUND ⚠️"}')

## 3. Load, Normalize & Clean Training Data

In [ ]:
def normalize_svg_to_256(svg_text):
    """Normalize SVG canvas to 256x256 using regex — no XML parsing required."""
    if not svg_text or not isinstance(svg_text, str):
        return None
    svg_text = svg_text.strip()
    if not svg_text.startswith('<svg'):
        return None
    if '</svg>' not in svg_text:
        return None
    if len(svg_text) > CONFIG['max_svg_chars']:
        return None

    # Regex-based canvas fix (doesn't require valid XML)
    # Replace or add width/height
    if re.search(r'width=', svg_text):
        svg_text = re.sub(r'width="[^"]*"', 'width="256"', svg_text, count=1)
        svg_text = re.sub(r"width='[^']*'", 'width="256"', svg_text, count=1)
    else:
        svg_text = svg_text.replace('<svg', '<svg width="256"', 1)

    if re.search(r'height=', svg_text):
        svg_text = re.sub(r'height="[^"]*"', 'height="256"', svg_text, count=1)
        svg_text = re.sub(r"height='[^']*'", 'height="256"', svg_text, count=1)
    else:
        svg_text = svg_text.replace('<svg', '<svg height="256"', 1)

    # Ensure xmlns
    if 'xmlns' not in svg_text:
        svg_text = svg_text.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)

    # Collapse whitespace
    svg_text = re.sub(r'\s+', ' ', svg_text).strip()

    return svg_text

In [ ]:
df = pd.read_csv(CONFIG['train_csv'])
print(f'Raw: {len(df)} rows')

valid_rows = []
reject_reasons = Counter()
for _, row in df.iterrows():
    svg = str(row['svg']).strip()
    prompt = str(row['prompt']).strip()
    if not prompt or len(prompt) < 5:
        reject_reasons['bad_prompt'] += 1; continue
    normalized = normalize_svg_to_256(svg)
    if normalized is None:
        reject_reasons['normalization_failed'] += 1; continue
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    valid_rows.append({'prompt': prompt, 'svg': normalized})

print(f'Clean: {len(valid_rows)} / {len(df)} ({100*len(valid_rows)/len(df):.1f}%)')
for r, c in reject_reasons.most_common(): print(f'  {r}: {c}')
assert 'width="256"' in valid_rows[0]['svg']
print('Canvas check ✓')

In [ ]:
random.shuffle(valid_rows)
n_eval = max(100, int(len(valid_rows) * CONFIG['eval_fraction']))
train_dataset = Dataset.from_list(valid_rows[n_eval:])
eval_dataset = Dataset.from_list(valid_rows[:n_eval])
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

## 4. Format for SFT

Qwen3.5 uses `<|im_start|>`/`<|im_end|>` chat template. No thinking mode.

In [ ]:
def format_chat(example):
    text = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{example["prompt"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{example["svg"]}<|im_end|>'
    )
    return {'text': text}

train_formatted = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
eval_formatted = eval_dataset.map(format_chat, remove_columns=eval_dataset.column_names)

# Filter too-long samples
print(f'Before filter: {len(train_formatted)}')
train_formatted = train_formatted.filter(
    lambda x: len(x['text']) < CONFIG['max_seq_length'] * 3
)
print(f'After filter: {len(train_formatted)}')
print(f'\nSample: {train_formatted[0]["text"][:400]}')

## 5. Load Model via Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=0,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps', eval_steps=CONFIG['eval_steps'],
    save_strategy='steps', save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    report_to='none', optim='paged_adamw_8bit', lr_scheduler_type='cosine',
    seed=SEED,
    max_length=CONFIG['max_seq_length'], packing=True, dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=train_formatted, eval_dataset=eval_formatted, args=sft_config,
)

eff_batch = CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']
print(f'Effective batch: {eff_batch} | Max steps: {trainer.state.max_steps}')

In [ ]:
t0 = time.time()
train_result = trainer.train()
elapsed = (time.time() - t0) / 60
print(f'\nTraining complete in {elapsed:.1f} minutes')
print(f'Final train loss: {train_result.training_loss:.4f}')

## 7. Save + Prepare for Inference

In [ ]:
adapter_dir = CONFIG['adapter_save_dir']
os.makedirs(adapter_dir, exist_ok=True)
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

summary = {
    'model': CONFIG['model_name'], 'lora_r': CONFIG['lora_r'],
    'epochs': CONFIG['num_train_epochs'], 'batch': eff_batch,
    'lr': CONFIG['learning_rate'], 'train_samples': len(train_formatted),
    'final_loss': train_result.training_loss, 'time_min': elapsed,
    'gpu': torch.cuda.get_device_name(0), 'seed': SEED,
}
with open(f'{PROJECT_DIR}/training_summary_v3_unsloth.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Summary: {summary}')

In [ ]:
# Switch to inference mode (Unsloth-specific)
FastLanguageModel.for_inference(model)

## 8. Inference

In [ ]:
def extract_svg(text):
    m = re.search(r'<svg[\s\S]*?</svg>', text, re.IGNORECASE)
    return m.group(0).strip() if m else ''

def fallback_svg(prompt):
    colors = ['red','blue','green','yellow','orange','purple','black','white','pink','brown','gray']
    fill = 'gray'
    for c in colors:
        if c in prompt.lower(): fill = c; break
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        f'<rect width="256" height="256" fill="white"/>'
        f'<circle cx="128" cy="128" r="64" fill="{fill}"/>'
        '</svg>'
    )

def strict_validate(svg):
    if not svg or len(svg) > 16000: return False
    try: root = ET.fromstring(svg)
    except ET.ParseError: return False
    root_tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
    if root_tag != 'svg': return False
    pc = 0
    for elem in root.iter():
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        if tag not in ALLOWED_TAGS: return False
        if tag == 'path': pc += 1
    return pc <= 256

def clean_and_validate_svg(svg):
    if not svg: return None
    ET.register_namespace('', 'http://www.w3.org/2000/svg')
    try: root = ET.fromstring(svg)
    except ET.ParseError: return None
    def remove_bad(elem):
        for child in list(elem):
            tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
            if tag not in ALLOWED_TAGS: elem.remove(child)
            else: remove_bad(child)
    remove_bad(root)
    root.set('xmlns', 'http://www.w3.org/2000/svg')
    root.set('width', '256'); root.set('height', '256')
    if 'viewBox' not in root.attrib: root.set('viewBox', '0 0 256 256')
    svg_out = ET.tostring(root, encoding='unicode')
    svg_out = svg_out.replace('ns0:', '').replace(':ns0', '')
    return svg_out if strict_validate(svg_out) else None

from transformers import AutoTokenizer

# Use the base text tokenizer, not the multimodal processor
text_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-2B")

# ── Optimized generate function ──
def generate_svg(prompt):
    messages = [
        {"role": "system", "content": "Generate SVG code."},
        {"role": "user", "content": prompt},
    ]
    text = text_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = text_tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,       # Was 768 — most SVGs finish under 512
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            repetition_penalty=1.4,
        )
    decoded = text_tokenizer.decode(output_ids[0], skip_special_tokens=False)
    assistant = decoded.split('<|im_start|>assistant\n')[-1]

    if '</think>' in assistant:
        assistant = assistant.split('</think>')[-1]
    for tok in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
        assistant = assistant.split(tok)[0]

    svg = extract_svg(assistant)

    if not svg and '<svg' in assistant:
        start = assistant.index('<svg')
        partial = assistant[start:].rstrip()
        last_sc = partial.rfind('/>')
        last_et = partial.rfind('</')
        if last_et != -1:
            try: partial = partial[:partial.index('>', last_et) + 1]
            except ValueError:
                if last_sc != -1: partial = partial[:last_sc + 2]
        elif last_sc != -1:
            partial = partial[:last_sc + 2]
        if '</svg>' not in partial: partial += '</svg>'
        svg = partial

    if svg:
        svg = re.sub(r"width=['\"][^'\"]*['\"]", 'width="256"', svg, count=1)
        svg = re.sub(r"height=['\"][^'\"]*['\"]", 'height="256"', svg, count=1)
        if 'viewBox' not in svg:
            svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
        if 'xmlns' not in svg:
            svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
        svg = svg.replace('""', '"')
        if len(svg) <= 16000 and svg.strip().startswith('<svg') and '</svg>' in svg:
            return svg

    return fallback_svg(prompt)

# ── Batched generation (processes multiple prompts at once) ──
def generate_batch(prompts, batch_size=4):
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        batch_texts = []
        for p in batch:
            messages = [
                {"role": "system", "content": "Generate SVG code."},
                {"role": "user", "content": p},
            ]
            batch_texts.append(text_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

        inputs = text_tokenizer(batch_texts, return_tensors='pt', padding=True).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.5,
                top_p=0.9,
                repetition_penalty=1.4,
            )

        for j, ids in enumerate(output_ids):
            decoded = text_tokenizer.decode(ids, skip_special_tokens=False)
            assistant = decoded.split('<|im_start|>assistant\n')[-1]
            if '</think>' in assistant:
                assistant = assistant.split('</think>')[-1]
            for tok in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
                assistant = assistant.split(tok)[0]

            svg = extract_svg(assistant)
            if not svg and '<svg' in assistant:
                start = assistant.index('<svg')
                partial = assistant[start:].rstrip()
                last_sc = partial.rfind('/>')
                last_et = partial.rfind('</')
                if last_et != -1:
                    try: partial = partial[:partial.index('>', last_et) + 1]
                    except ValueError:
                        if last_sc != -1: partial = partial[:last_sc + 2]
                elif last_sc != -1:
                    partial = partial[:last_sc + 2]
                if '</svg>' not in partial: partial += '</svg>'
                svg = partial

            if svg:
                svg = re.sub(r"width=['\"][^'\"]*['\"]", 'width="256"', svg, count=1)
                svg = re.sub(r"height=['\"][^'\"]*['\"]", 'height="256"', svg, count=1)
                if 'viewBox' not in svg:
                    svg = svg.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
                if 'xmlns' not in svg:
                    svg = svg.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
                svg = svg.replace('""', '"')
                if len(svg) <= 16000 and svg.strip().startswith('<svg') and '</svg>' in svg:
                    results.append(svg)
                    continue
            results.append(fallback_svg(batch[j]))
    return results

In [ ]:
# ── Reinstall & imports ──
!pip install -q transformers accelerate peft bitsandbytes pandas lxml
!pip install -q --upgrade transformers

import os, re, time, json
import xml.etree.ElementTree as ET
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ── Mount Drive ──
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/svg-competition'

# ── Load model + adapter ──
print("Loading model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

# Text-only base model (avoids Qwen3.5 vision issue)
text_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-2B")
text_tokenizer.padding_side = 'left'  # Add this line
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-2B",
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

# Load your trained adapter from Drive
ADAPTER_PATH = f'{PROJECT_DIR}/svg-lora-adapter-v3-unsloth'
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model = model.merge_and_unload()
model.eval()
print(f"Model ready on {model.device}")

In [ ]:
test_df = pd.read_csv(f'{PROJECT_DIR}/test.csv')
rows = []
fallback_count = 0
t0 = time.time()

for idx, row in test_df.iterrows():
    prompt = str(row['prompt']).strip()
    t1 = time.time()
    svg = generate_svg(prompt)
    gen_time = time.time() - t1
    is_fb = len(svg) < 190
    if is_fb: fallback_count += 1
    rows.append({'id': row['id'], 'svg': svg})
    print(f'  [{idx+1}/{len(test_df)}] {gen_time:.1f}s | len={len(svg)} | fb={is_fb} | total_fb={fallback_count}')

elapsed_total = (time.time() - t0) / 60
print(f'\nDone! {len(rows)} SVGs in {elapsed_total:.1f} min')
print(f'Fallbacks: {fallback_count}/{len(rows)} ({100*fallback_count/len(rows):.1f}%)')

sub_df = pd.DataFrame(rows)
SUBMISSION_PATH = f'{PROJECT_DIR}/submission_v3.csv'
sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f'Saved: {SUBMISSION_PATH}')

from google.colab import files
files.download(SUBMISSION_PATH)

## AI Tooling Disclosure

- **Claude (Anthropic)**: Coding assistance, debugging.